In [2]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from minatar import Environment 
import cv2
from stable_baselines3 import DQN
from sb3_contrib import QRDQN, TQC
from stable_baselines3.common.evaluation import evaluate_policy

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
class MinAtarEnv(gym.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(self, game="breakout"):
        super().__init__()
        self.env = Environment(env_name=game)

        obs_shape = self.env.state_shape()
        self.observation_space = spaces.Box(
            low=0, high=1, shape=obs_shape, dtype=np.float32
        )

        self.action_space = spaces.Discrete(self.env.num_actions())

    def reset(self, seed=None, options=None):
        if seed is not None:
            np.random.seed(seed)
        self.env.reset()
        return self.env.state(), {}

    def step(self, action):
        reward, done = self.env.act(action)
        return self.env.state(), reward, done, False, {}

    def render(self):
        self.env.display_state()

In [4]:
class BlurWrapper(gym.ObservationWrapper):
    def __init__(self, env, blur=True, region=(0, 0, 5, 5), ksize=3):
        super().__init__(env)
        self.blur = blur
        self.x1, self.y1, self.x2, self.y2 = region
        self.ksize = ksize

    def observation(self, obs):
        if not self.blur:
            return obs

        obs = obs.copy()
        patch = obs[self.y1 : self.y2, self.x1 : self.x2, :]
        patch = cv2.GaussianBlur(patch, (self.ksize, self.ksize), 0)
        obs[self.y1 : self.y2, self.x1 : self.x2, :] = patch
        return obs

In [5]:
def make_breakout_env(blur=False):
    env = MinAtarEnv("breakout")
    env = BlurWrapper(env, blur=blur, region=(0, 0, 10, 4), ksize=3)
    return env

In [6]:
def build_model(algo, env):
    if algo == "dqn":
        return DQN("MlpPolicy", env, verbose=1)
    elif algo == "rainbow":
        return QRDQN("MlpPolicy", env, verbose=1)
    elif algo == "c51":
        return DQN("MlpPolicy", env, verbose=1, policy_kwargs=dict(n_atoms=51))
    elif algo == "sac_discreto":
        return TQC("MlpPolicy", env, verbose=1)
    else:
        raise ValueError("Algoritmo inválido.")

In [7]:
def run_experiment_breakout(algo="dqn", blur=False):

    env = make_breakout_env(blur=blur)
    model = build_model(algo, env)

    print(f"\nTreinando {algo} | blur={blur}")
    model.learn(total_timesteps=50_000)

    mean_reward, std = evaluate_policy(model, env, n_eval_episodes=20)

    print(f"Reward médio: {mean_reward}")
    return mean_reward

In [8]:
results = {}

for algo in ["dqn"]:
    results[(algo, "sem_blur")] = run_experiment_breakout(algo, blur=False)
    results[(algo, "com_blur")] = run_experiment_breakout(algo, blur=True)

results

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.

Treinando dqn | blur=False
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 16       |
|    ep_rew_mean      | 1        |
|    exploration_rate | 0.988    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 10958    |
|    time_elapsed     | 0        |
|    total_timesteps  | 64       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 11       |
|    ep_rew_mean      | 0.5      |
|    exploration_rate | 0.983    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 9866     |
|    time_elapsed     | 0        |
|    total_timesteps  | 88       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 12       |
| 

KeyboardInterrupt: 